In [ ]:
import math
import numpy as np 
import matplotlib.pyplot as plt
import random

In [ ]:
def f(x):
    return 3*(x**2) - 4*x + 5 

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
xs

In [ ]:
ys = f(xs)
plt.plot(xs, ys)

how does f(x) respond, when changing x moves upwards or downwards

$
f{'}(x) = \frac{f(x + h) - f(x)}{h}
$

In [ ]:
h = 0.000001
x = 2/3
(f(x + h) - f(x)) / h

In [ ]:
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
d

In [ ]:
h = 0.0001

#inputs
a = 2.0
b = -3.0
c = 10.0

d1 = a*b + c
c += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2-d1) / h)

_backward() returns nothing but updates .grad

In [ ]:
class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
      
    return out
  
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out
  
  def __rmul__(self, other): # other * self
    return self * other

  def __truediv__(self, other): # self / other
    return self * other**-1

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  def __radd__(self, other): # other + self
    return self + other

  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')
    
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    
    return out
  
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')
    
    def _backward():
      self.grad += out.data * out.grad # NOTE: in the video I incorrectly used = instead of +=. Fixed here.
    out._backward = _backward
    
    return out
  
  
  def backward(self):
    
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
d._prev

In [ ]:
d._op

In [ ]:
from graphviz import Digraph

def trace(root):
# builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
    # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
draw_dot(L)

Chain Rule 

$ \frac{dL}{dc} = \frac{dL}{dd} \cdot \frac{dd}{dc} $

$$
\begin{aligned}
\frac{dL}{de} = -2.0 \\

\frac{dL}{da} = \frac{dL}{de} \cdot \frac{de}{da} \\
\frac{dL}{da}  = -2.0 \cdot \frac{de}{da} \\

a = 2.0, b = -3.0 \\
e = a \cdot b \\
\frac{de}{da} = b \cdot \frac{de}{da}a \\
\frac{de}{da} = b \cdot 1 \\
\frac{de}{da} = -3.0 \\

\frac{dL}{da} = -2.0 \cdot -3.0 \\
\frac{dL}{da} = 6.0
\end{aligned}
$$

In [ ]:
def lol():
    h = 0.001
    
    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b; e.label='e'
    d = e + c; d.label='d'
    f = Value(-2.0, label='f')
    L = d * f; L.label='L'
    L1 = L.data
    
    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b; e.label='e'
    d = e + c; d.label='d'
    e.data += h
    f = Value(-2.0, label='f')
    L = d * f; L.label='L'
    L2 = L.data
    
    print((L2 - L1) / h)
    
lol()

In [ ]:
plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2)))
plt.grid()

$$
tanh x = \frac{sinh x}{cosh x} = \frac{e^{x} - e^{-x}}{e^{x} - e^{-x}} = \frac{e^{2x} - 1}{e^{2x} + 1}
$$

$ \frac{d}{dx}tanh = 1 - tanh^{2}x$

In [ ]:
2, 0, -3, 1, 6.7

# inputs
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

#weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

#bias
b = Value(6.8813735870195432, label='b')

# (x1*w1) + (x2*w2) + b
x1w1 = x1 * w1; x1w1.label = 'x1w1'
x2w2 = x2 * w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1x2w2'
n = x1w1x2w2 + b; n.label = 'n'

o = n.tanh(); o.label = 'o'

In [ ]:
draw_dot(o)

In [ ]:
o.backward()

In [ ]:
o.grad = 1.0

topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)
build_topo(o)
topo

for node in reversed(topo):
    node._backward()

In [ ]:
o.grad = 1.0
o._backward()
n._backward()
b._backward()
x1w1x2w2._backward()
x1w1._backward()
x2w2._backward()
x2._backward()
w2._backward()
x1._backward()
w1._backward()

In [ ]:
o.grad = 1.0
n.grad = 1 - (o.data)**2
b.grad = 0.5
x1w1x2w2.grad = 0.5
x1w1.grad = 0.5
x2w2.grad = 0.5
x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad
x1.grad = w1.data * x1w1.grad 
w1.grad = x1.data * x1w1.grad

In [ ]:
2, 0, -3, 1, 6.7

# inputs
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

#weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

#bias
b = Value(6.8813735870195432, label='b')

# (x1*w1) + (x2*w2) + b
x1w1 = x1 * w1; x1w1.label = 'x1w1'
x2w2 = x2 * w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1x2w2'
n = x1w1x2w2 + b; n.label = 'n'

# Manual Tanh
e = (2*n).exp()
o = (e-1) / (e+1); o.label = 'o'

# Drawing
o.label = 'o'
o.backward()
draw_dot(o)

In [ ]:
import torch 

x1 = torch.Tensor([2.0]).double(); x1.requires_grad = True
x2 = torch.Tensor([0.0]).double(); x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True
w2 = torch.Tensor([1.0]).double(); w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('----')

print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

In [161]:
class Neuron:
    
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
        
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
class Layer:
    
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin=nin) for _ in range(nout)]
        
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs
    
class MLP:
    
    def __init__(self, nin, nouts):
        size = [nin] + nouts
        self.layers = [Layer(nin=size[i], nout=size[i+1]) for i in range(len(nouts))] # creates [Layer(a, b), Layer(b, c), Layer(c, d)]
        
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x) # Layer.__call__(x) here
        return x

xs = [2.0, 3.0, -1.0]

n = Neuron(nin=2)
print(n(xs))

l = Layer(nin=2, nout=3)
print(l(xs))

mlp = MLP(nin=3, nouts=[4, 4, 1])
print(mlp(xs))

Value(data=0.914428090862905)
[Value(data=0.9979463733227246), Value(data=0.9931509478545024), Value(data=0.9151691131546977)]
[Value(data=-0.9869460946099436)]
